# Real-data audit before Pillar-0
Attach the competition data and this working project as a private Kaggle input. No image download to your machine is needed. Set the actual paths below. This notebook does not train or extract embeddings.


In [ ]:
from pathlib import Path
import sys, subprocess
PROJECT = Path("/kaggle/input/YOUR_PRIVATE_CODE_INPUT/repository")
DATA = Path("/kaggle/input/YOUR_COMPETITION_INPUT")
OUTPUT = Path("/kaggle/working/knee-audit")
assert (PROJECT / "pyproject.toml").is_file(), "Set PROJECT to this working project"
assert DATA.is_dir(), "Set DATA to the mounted competition directory"
subprocess.run([sys.executable, "-m", "pip", "install", str(PROJECT)], check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "rsna_knee.data.audit", str(DATA), "--config", str(PROJECT / "configs/audit.json"), "--output", str(OUTPUT)], check=True)


In [ ]:
import json
from IPython.display import Markdown, display
display(Markdown((OUTPUT / "AUDIT.md").read_text()))
audit = json.loads((OUTPUT / "audit.json").read_text())
for item in audit["sampled_series"]:
    print(item["directory"], item["status"], item.get("error", ""))


Review table schemas before identifying gold labels; do not use reports as binary targets. Inspect a few valid sampled series below. Native axes and LPS coordinates are retained; these views must be reviewed for real anatomy and ordering. Failures are saved and displayed, never substituted.


In [ ]:
import numpy as np
from rsna_knee.preprocessing.dicom import reconstruct, visualize
from IPython.display import Image
valid = [x for x in audit["sampled_series"] if x["status"] == "valid_geometry"]
for index, item in enumerate(valid[:3]):
    folder = DATA / item["directory"]
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in audit["config"]["dicom_suffixes"])
    try:
        volume, metadata = reconstruct(files)
        dest = OUTPUT / f"volume-{index}"
        dest.mkdir(exist_ok=True)
        np.save(dest / "volume.npy", volume)
        (dest / "geometry.json").write_text(json.dumps(metadata, indent=2))
        visualize(volume, metadata, dest / "views.png")
        display(Image(filename=str(dest / "views.png")))
    except Exception as exc:
        error = {"directory": item["directory"], "error": f"{type(exc).__name__}: {exc}"}
        (OUTPUT / f"volume-{index}-error.json").write_text(json.dumps(error, indent=2))
        print(error)


Keep artifacts private. Review audit results and these images before proceeding to the checkpoint/preprocessing integration. This notebook has not been executed on competition data by the implementation agent.
